In [ ]:
import cv2
import numpy as np
import os
from PIL import Image
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from torch.nn.parallel import DataParallel
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
from torchvision.utils import save_image

from tqdm import tqdm
from models.srcnn import *
from models.vdsr import *

from models.sr_model import *
from models.hqsr import *
from models.vdsr import *
from models.srresnet import *
from models.sr_model import *
from models.vdsr import *
from models.utils import *
from models.srcnn import *
from models.edsr import *
import time
from models.e2dsr import *
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device('cpu')
t = 33

In [ ]:
from torchmetrics import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

def calculate_metrics(img1, img2, max_pixel_value=1.0):
    psnr = PeakSignalNoiseRatio(data_range=1.0).to(device)
    ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    psnr_value = psnr(img1, img2)
    ssim_value = ssim(img1, img2)
    return psnr_value, ssim_value

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, type =False, scale = 4):
        self.lr_files = sorted(os.listdir(lr_dir))
        self.hr_files = sorted(os.listdir(hr_dir))
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.type =type
        self.scale = scale
    def __len__(self):
        return len(self.lr_files)

    def __getitem__(self, idx):
        lr_image = Image.open(os.path.join(self.lr_dir, self.lr_files[idx])).convert('RGB')
        hr_image = Image.open(os.path.join(self.hr_dir, self.hr_files[idx])).convert('RGB')
        width, height = hr_image.size
        lr_height, lr_width = height // self.scale, width // self.scale
        hr_height, hr_width = lr_height * self.scale, lr_width*self.scale
        lr_image = lr_image.resize((lr_width, lr_height))
        hr_image = hr_image.resize((hr_width, hr_height))
        if self.type:
            lr_image = lr_image.resize((hr_width, hr_height))
            
        # hr_image = cv2.resize(hr_image, (hr_width, hr_height), interpolation=cv2.INTER_CUBIC)

        transform = transforms.Compose([
            # transforms.ToPILImage(),
            transforms.ToTensor()
        ])
        
        lr_image = transform(lr_image)
        hr_image = transform(hr_image)
        return lr_image, hr_image
    

In [ ]:
import time
        # sub = 'Set14'
# for scale in[2, 3, 4]:
for scale in [4]:
    # edsr_srcnn = EDSR_srcnnfy()
    # edsr  = EDSR_orig().to(device)
    # scale = 4
    print(f'scale {scale}')
    log_file = open('outputs/test_log/subtracttion.txt', 'a')
    log_file.write(f'scale {scale}')
    edsr = EDSR(scale).to(device)
    edrn_canny = HQSR(scale_factor=scale, use_canny=True).to(device)
    edrn_sobel = HQSR(scale_factor=scale,use_sobel=True).to(device)
    srresnet = SRResNet(scale=scale).to(device)
    vdsr = VDSR().to(device)
    e2dsr_sobel = E2DSR(scale_factor=scale, edge_option='sobel').to(device)
    e2dsr_canny = E2DSR(scale_factor=scale,edge_option='canny').to(device)
    srcnn = SRCNN().to(device)

    edsr.load_state_dict(torch.load(f'outputs/weight_sr/x{scale}/best_edsr.pth', map_location=device))
    # edrn_sobel.load_state_dict(torch.load(f'outputs/weight_sr/x{scale}/best_hqsr_sobel.pth', map_location=device))
    # edrn_canny.load_state_dict(torch.load(f'outputs/weight_sr/x{scale}/best_hqsr_canny.pth', map_location=device))
    srresnet.load_state_dict(torch.load(f'outputs/weight_sr/x{scale}/best_srresnet.pth', map_location=device))
    vdsr.load_state_dict(torch.load(f'outputs/weight_sr/x{scale}/best_vdsr.pth', map_location=device))
    srcnn.load_state_dict(torch.load(f'outputs/weight_sr/x{scale}/best_srcnn.pth', map_location=device))
    e2dsr_canny.load_state_dict(torch.load(f'outputs/weight_sr/x{scale}/best_e2dsr_canny.pth', map_location=device))
    e2dsr_sobel.load_state_dict(torch.load(f'outputs/weight_sr/x{scale}/best_e2dsr_sobel.pth', map_location=device))
    # for sub in ['BSDS100', 'DIV2K', 'Set5', 'Set14', 'Urban100']:
    data_path = 'tên của bộ dữ liệu, nên đưa vào trong folder benchmark'
    for sub in [data_path]:
        print(f'{sub}\n')
        valid_lr_dir = f'benchmark/{sub}/HR'
        valid_hr_dir = f'benchmark/{sub}/HR'
        output_image_dir = f'benchmark/output/{sub}'
        valid_dataset = ImageDataset(valid_lr_dir, valid_hr_dir,scale=scale)
        valid_dataset_2 = ImageDataset(valid_lr_dir, valid_hr_dir, type = True, scale=scale)
        valid_loader = DataLoader(valid_dataset)
        valid_loader_2 = DataLoader(valid_dataset_2)
        log_file.write(f'{sub}\n')
        # Đo thời gian xử lý trung bình cho từng mô hình
        e2dsr_sobel_total_time = 0
        e2dsr_canny_total_time = 0
        sobel_total_time = 0
        canny_total_time = 0
        edsr_total_time = 0
        srresnet_total_time = 0
        vdsr_total_time = 0
        srcnn_total_time = 0
        bicubic_total_time = 0

        edrn_sobel.eval()
        edrn_canny.eval()
        srresnet.eval()
        srcnn.eval()
        edsr.eval()
        vdsr.eval()
        e2dsr_canny.eval()
        e2dsr_sobel.eval()

        val_psnr_values_bicubic = 0
        val_psnr_values_sobel = 0
        val_psnr_values_canny = 0
        val_psnr_values_edsr = 0
        val_psnr_values_srresnet = 0
        val_psnr_values_vdsr = 0
        val_psnr_values_srcnn = 0
        val_psnr_values_e2dsr_canny = 0
        val_psnr_values_e2dsr_sobel = 0

        val_ssim_values_sobel = 0
        val_ssim_values_canny = 0
        val_ssim_values_edsr = 0
        val_ssim_values_srresnet = 0
        val_ssim_values_vdsr = 0
        val_ssim_values_srcnn = 0
        val_ssim_values_bicubic = 0
        val_ssim_values_e2dsr_canny = 0
        val_ssim_values_e2dsr_sobel = 0

        torch.cuda.empty_cache()
        with torch.no_grad():  # Không tính toán gradient trong quá trình validation
            for (lr_images, hr_images) in tqdm(valid_loader_2, unit='batch'):
                lr_images = lr_images.to(device)
                hr_images = hr_images.to(device)
                # VDSR validation
                start_time = time.time()
                outputs_vdsr = vdsr(lr_images)
                vdsr_total_time += time.time() - start_time
                psnr_vdsr, ssim_vdsr = calculate_metrics(outputs_vdsr, hr_images)

                # SRCNN validation
                start_time = time.time()
                outputs_srcnn = srcnn(lr_images)
                srcnn_total_time += time.time() - start_time
                psnr_srcnn, ssim_srcnn = calculate_metrics(outputs_srcnn, hr_images)

                # Cập nhật PSNR và SSIM cho VDSR và SRCNN
                val_psnr_values_vdsr += psnr_vdsr
                val_psnr_values_srcnn += psnr_srcnn

                val_ssim_values_vdsr += ssim_vdsr
                val_ssim_values_srcnn += ssim_srcnn

            # Tính PSNR và SSIM trung bình cho VDSR và SRCNN
            val_average_psnr_vdsr = val_psnr_values_vdsr / len(valid_loader_2)
            val_average_psnr_srcnn = val_psnr_values_srcnn / len(valid_loader_2)

            val_average_ssim_vdsr = val_ssim_values_vdsr / len(valid_loader_2)
            val_average_ssim_srcnn = val_ssim_values_srcnn / len(valid_loader_2)

            # Thời gian xử lý trung bình cho VDSR và SRCNN
            avg_time_vdsr = vdsr_total_time / len(valid_loader_2)
            avg_time_srcnn = srcnn_total_time / len(valid_loader_2)

            # Ghi kết quả vào log_file
            
        torch.cuda.empty_cache()
        with torch.no_grad():  # Không tính toán gradient trong quá trình validation
            for (lr_images, hr_images) in tqdm(valid_loader, unit='batch'):
                lr_images = lr_images.to(device)
                hr_images = hr_images.to(device)

                # Sobel SR validation (không tính loss, chỉ tính PSNR và SSIM)
                start_time = time.time()  # Đo thời gian bắt đầu
                outputs_sobel = edrn_sobel(lr_images)
                sobel_total_time += time.time() - start_time  # Tính thời gian xử lý
                psnr_sobel, ssim_sobel = calculate_metrics(outputs_sobel, hr_images)

                #bicubic
                start_time = time.time()  # Đo thời gian bắt đầu
                outputs_bicubic = F.interpolate(lr_images, scale_factor=scale, mode='bilinear', align_corners=False)
                bicubic_total_time += time.time() - start_time  # Tính thời gian xử lý bicubic
                psnr_bicubic, ssim_bicubic = calculate_metrics(outputs_bicubic, hr_images)

                # Canny SR validation
                start_time = time.time()
                outputs_canny = edrn_canny(lr_images)
                canny_total_time += time.time() - start_time
                psnr_canny, ssim_canny = calculate_metrics(outputs_canny, hr_images)

                # Canny SR validation
                start_time = time.time()
                outputs_e2dsr_canny = e2dsr_canny(lr_images)
                e2dsr_canny_total_time += time.time() - start_time
                psnr_e2dsr_canny, ssim_e2dsr_canny = calculate_metrics(outputs_e2dsr_canny, hr_images)

                # Canny SR validation
                start_time = time.time()
                outputs_e2dsr_sobel = e2dsr_sobel(lr_images)
                e2dsr_sobel_total_time += time.time() - start_time
                psnr_e2dsr_sobel, ssim_e2dsr_sobel = calculate_metrics(outputs_e2dsr_sobel, hr_images)

                # EDSR SR validation
                start_time = time.time()
                outputs_edsr = edsr(lr_images)
                edsr_total_time += time.time() - start_time
                psnr_edsr, ssim_edsr = calculate_metrics(outputs_edsr, hr_images)

                # SRResNet SR validation
                start_time = time.time()
                outputs_srresnet = srresnet(lr_images)
                srresnet_total_time += time.time() - start_time
                psnr_srresnet, ssim_srresnet = calculate_metrics(outputs_srresnet, hr_images)

                # Cập nhật PSNR và SSIM
                val_psnr_values_sobel += psnr_sobel
                val_psnr_values_canny += psnr_canny
                val_psnr_values_edsr += psnr_edsr
                val_psnr_values_srresnet += psnr_srresnet
                val_psnr_values_bicubic += psnr_bicubic
                val_psnr_values_e2dsr_canny += psnr_e2dsr_canny
                val_psnr_values_e2dsr_sobel += psnr_e2dsr_sobel

                val_ssim_values_sobel += ssim_sobel
                val_ssim_values_canny += ssim_canny
                val_ssim_values_edsr += ssim_edsr
                val_ssim_values_srresnet += ssim_srresnet
                val_ssim_values_bicubic += ssim_bicubic
                val_ssim_values_e2dsr_canny += ssim_e2dsr_canny
                val_ssim_values_e2dsr_sobel += ssim_e2dsr_sobel
            # Tính PSNR và SSIM trung bình
            val_average_psnr_sobel = val_psnr_values_sobel / len(valid_loader)
            val_average_psnr_canny = val_psnr_values_canny / len(valid_loader)
            val_average_psnr_edsr = val_psnr_values_edsr / len(valid_loader)
            val_average_psnr_srresnet = val_psnr_values_srresnet / len(valid_loader)
            val_average_psnr_bicubic = val_psnr_values_bicubic / len(valid_loader)
            val_average_psnr_e2dsr_canny = val_psnr_values_e2dsr_canny/len(valid_loader)
            val_average_psnr_e2dsr_sobel  = val_psnr_values_e2dsr_sobel / len(valid_loader)

            
            val_average_ssim_sobel = val_ssim_values_sobel / len(valid_loader)
            val_average_ssim_canny = val_ssim_values_canny / len(valid_loader)
            val_average_ssim_edsr = val_ssim_values_edsr / len(valid_loader)
            val_average_ssim_srresnet = val_ssim_values_srresnet / len(valid_loader)
            val_average_ssim_bicubic = val_ssim_values_bicubic / len(valid_loader)
            val_average_ssim_e2dsr_canny = val_ssim_values_e2dsr_canny/ len(valid_loader)
            val_average_ssim_e2dsr_sobel = val_ssim_values_e2dsr_sobel/len(valid_loader)
            # Thời gian xử lý trung bình
            avg_time_sobel = sobel_total_time / len(valid_loader)
            avg_time_canny = canny_total_time / len(valid_loader)
            avg_time_edsr = edsr_total_time / len(valid_loader)
            avg_time_srresnet = srresnet_total_time / len(valid_loader)
            avg_time_bicubic = bicubic_total_time / len(valid_loader)
            avg_time_e2dsr_canny = e2dsr_canny_total_time/len(valid_loader)
            avg_time_e2dsr_sobel = e2dsr_sobel_total_time/len(valid_loader)

            # Thời gian xử lý trung bình cho Bicubic
            
            temp_psnr = val_average_psnr_bicubic
            temp_ssim = val_average_ssim_bicubic
            # Ghi kết quả vào log_file
            # Validation cho VDSR và SRCNN
        log_file.write(f'Bicubic:  SSIM / PSNR {val_average_ssim_bicubic:.4f} / {val_average_psnr_bicubic:.2f}, Time {avg_time_bicubic:.4f}s\n')
        log_file.write(f'VDSR:  SSIM / PSNR {val_average_ssim_vdsr:.4f} / {val_average_psnr_vdsr:.2f}, Time {avg_time_vdsr:.4f}s\n')
        log_file.write(f'SRCNN:  SSIM / PSNR {val_average_ssim_srcnn:.4f} / {val_average_psnr_srcnn:.2f}, Time {avg_time_srcnn:.4f}s\n')    
        log_file.write(f'SRResNet:  SSIM / PSNR {val_average_ssim_srresnet:.4f} / {val_average_psnr_srresnet:.2f}, Time {avg_time_srresnet:.4f}s\n')
        log_file.write(f'EDSR:  SSIM / PSNR {val_average_ssim_edsr:.4f} / {val_average_psnr_edsr:.2f}, Time {avg_time_edsr:.4f}s\n')
        log_file.write(f'Sobel:  SSIM / PSNR {val_average_ssim_sobel:.4f} / {val_average_psnr_sobel:.2f}, Time {avg_time_sobel:.4f}s\n')
        log_file.write(f'Canny:  SSIM / PSNR {val_average_ssim_canny:.4f} / {val_average_psnr_canny:.2f}, Time {avg_time_canny:.4f}s\n')
        log_file.write(f'E2DSR Sobel:  SSIM / PSNR {val_average_ssim_e2dsr_sobel:.4f} / {val_average_psnr_e2dsr_sobel:.2f}, Time {avg_time_e2dsr_sobel:.4f}s\n')
        log_file.write(f'E2DSR Canny:  SSIM / PSNR {val_average_ssim_e2dsr_canny:.4f} / {val_average_psnr_e2dsr_canny:.2f}, Time {avg_time_e2dsr_canny:.4f}s\n\n')
        log_file.flush()
        log_file.close
